# First EventDialCache entry

Load a GUNDAM engine and inspect the first entry of the model propagator `EventDialCache`.

In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Toy"  # "Asimov", "Toy", or "RealData"
seed = 12345

In [2]:
import sys
from pathlib import Path

import numpy as np

repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))

from gundam_interface import GundamInterface, GundamLoader, GundamRuntime

In [3]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)

{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Toy',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'randomSeed': 12345,
 'configPath': 'configOA2024.yaml',
 'overrideList': ['override/v12ProdRun45.yaml',
  'override/onlyFlux5.yaml',
  'override/noEigen.yaml']}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()
gundam.initialize()

2026.06.30 18:01:21  INFO ConfigUtils: Reading config file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/configOA2024.yaml
2026.06.30 18:01:21  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/v12ProdRun45.yaml"
2026.06.30 18:01:21  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/mc/filePathList
2026.06.30 18:01:21  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/data/0(name:"data")/filePathList
2026.06.30 18:01:21  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/onlyFlux5.yaml"
2026.06.30 18:01:21  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSetListConfig/0(name:"Flux Systematics")/isEnabled: true -> false
2026.06.30 18:01:21  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSe

In [5]:
def get_or_attr(obj, getter_name, attr_name):
    getter = getattr(obj, getter_name, None)
    if getter is not None:
        return getter()
    return getattr(obj, attr_name)

In [6]:
propagator = gundam.engine.getLikelihoodInterface().getModelPropagator()

eventDialCache = propagator.getEventDialCache()
cache = eventDialCache.getCache()

print(f"Number of cached entries: {len(cache)}")

Number of cached entries: 416034


In [7]:
entry = cache[0]

event = get_or_attr(entry, "getEvent", "event")
indices = event.getIndices()

print("Event")
print(f"  sample: {indices.sample}")
print(f"  entry:  {indices.entry}")
print(f"  bin:    {indices.bin}")
print(f"  weight: {event.getEventWeight():.8g}")

Event
  sample: 0
  entry:  14
  bin:    100
  weight: 0.050692335


In [8]:
dialResponseCacheList = get_or_attr(
    entry,
    "getDialResponseCacheList",
    "dialResponseCacheList",
)

print(f"Number of dial responses: {len(dialResponseCacheList)}")

for index, dialResponse in enumerate(dialResponseCacheList):
    dial = get_or_attr(dialResponse, "getDialInterface", "dialInterface")
    print(f"\nDial response {index}")
    print(dial.getSummary())
    print(dialResponse.getResponse())

Number of dial responses: 1

Dial response 0
Norm: input({ Flux Systematics/#5=1 }) applyOn(#5: isRHC: [0], NeutrinoCode: [14], Enu: [1, 1.5[) supervisor(minResponse=0) response=1
1.0
